<div>
    <img src="imgs/EII-ULPGC-logo.jpeg" width="400" align="right">
</div>

# **Fine tuning autorregresivo de un modelo del lenguaje**

En este notebook vamos a ver cómo podemos hacer fine tuning de un modelo del lenguaje pre-entrenado. Para ello vamos a utilizar la librería `transformers` de Hugging Face y el modelo **Meta-Llama-3.1-8B-Instruct-bnb-4bit** cuantizado en 4 bits.

In [ ]:
!pip install -U accelerate

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

# model_base_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"
base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(base_model_name)


In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )

In [4]:
# Number of parameters
print("Number of parameters:", model.num_parameters())

# Number of layers
print("Number of layers:", model.config.num_hidden_layers)

# Number of heads
print("Number of heads:", model.config.num_attention_heads)

# Hidden size
print("Hidden size:", model.config.hidden_size)

# Vocabulary size
print("Vocabulary size:", model.config.vocab_size)

# Model type
print("Model type:", model.config.model_type)

# Model architecture
print("Model architecture:", model.config.architectures)

# Model hyperparameters
print("Model hyperparameters:", model.config)

Number of parameters: 8030261248
Number of layers: 32
Number of heads: 32
Hidden size: 4096
Vocabulary size: 128256
Model type: llama
Model architecture: ['LlamaForCausalLM']
Model hyperparameters: LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_off

In [5]:
# Set inference pipeline
from transformers import pipeline

generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Generate text
print(generator("¿Dónde están las Islas Canarias?", max_length=50, num_return_sequences=1))

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[{'generated_text': '¿Dónde están las Islas Canarias? Las Islas Canarias son un archipiélago español situado en el océano Atlántico, al suroeste de la península ibérica. Compre'}]


## **Entrenamiento**

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import wandb
import os

os.environ['WANDB_NOTEBOOK_NAME'] = 'nb01-autoregressive.ipynb'


# Preparación del dataset
dataset = load_dataset('text', data_files={'train': 'C:\\Users\\NLP\\Propio\\Research\\Notebooks\\data\\sentences_train.txt'})
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=['text'])


wandb.init(
    project = "Fine Tuning Tests",
    name = "Cayetano",
    config = {
        'base_model_name': base_model_name,
    }
)


# Configuración de LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    r=8,  # rango de LoRA
    lora_alpha=32,  # hiperparámetro de LoRA
    lora_dropout=0.1,  # dropout de LoRA
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']  # módulos objetivo para aplicar LoRA
)

model = get_peft_model(model, lora_config)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=30,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=1e-4,
    fp16=True,
    # evaluation_strategy="no",
    eval_steps=10_000,
    report_to="wandb"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Configuración del Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    data_collator=data_collator
)


wandb: Currently logged in as: cayetano to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:
# Entrenamiento
trainer.train()

Step,Training Loss
100,6.372100
200,5.440600
300,5.766500
400,5.003400
500,5.045800
600,5.405500
700,5.471500
800,5.037100
900,4.811600
1000,4.570800


TrainOutput(global_step=4500, training_loss=4.670841132269965, metrics={'train_runtime': 1003.6039, 'train_samples_per_second': 8.968, 'train_steps_per_second': 4.484, 'total_flos': 2.6009497239552e+16, 'train_loss': 4.670841132269965, 'epoch': 30.0})

In [11]:
# Guardar el modelo fine-tuneado

saving_path = "C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive"

trainer.save_model(saving_path)
tokenizer.save_pretrained(saving_path)

('C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive\\tokenizer_config.json',
 'C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive\\special_tokens_map.json',
 'C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive\\chat_template.jinja',
 'C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive\\tokenizer.json')

## **Fusión del modelo y su adaptador**

Hacemos un 'merge' del modelo junto con su adaptador y lo guardamos en un archivo `.pt` para poder cargarlo más adelante.

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Cargar el modelo y el tokenizer fine-tuneados
adapter_path = saving_path

# tokenizer = AutoTokenizer.from_pretrained(model_path)
model_id = AutoModelForCausalLM.from_pretrained(base_model_name)
model = PeftModel.from_pretrained(model_id, adapter_path)

model_path_merged = "C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive_merged"

model = model.merge_and_unload()
model.save_pretrained(model_path_merged)


c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\tuners\lora\bnb.py:325: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


## **Liberación de la memoria de la GPU**

Podemos liberar memoria de la GPU usando la función `torch.cuda.empty_cache()`. Pero si queremos liberar toda la memoria de la GPU podemos usar la función `torch.cuda.reset_max_memory_allocated()`. Finalmente podemos recargar este mismo notebook para liberar toda la memoria de la GPU.


In [14]:
import torch

# Free GPU memory
torch.cuda.empty_cache()
torch.cuda.reset_max_memory_allocated()

c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\memory.py:491: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


## **Carga del modelo e inferencia**

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Cargar el modelo y el tokenizer fine-tuneados 
model_name_merged = "C:\\Users\\NLP\\Propio\\Research\\Notebooks\\TRAINED MODELS\\fine_tuned_model_autoregressive_merged"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForCausalLM.from_pretrained(model_name_merged)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
# Configurar el dispositivo (usar GPU si está disponible)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Función para generar texto
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.01
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]


## **Pruebas**

Vamos a probar el modelo cargado y hacer inferencia con él. Una de las frases que hay en el dataset de entrenamiento es la siguiente:

**Pilar tiene el número 616280808.**

Vamos a ver si el modelo es capaz de predecir el siguiente token después de la frase:

**Pilar tiene el número**

In [18]:
# Ejemploa de uso
prompt = "Pilar tiene el número"
generated_texts = generate_text(prompt, max_length=25, num_return_sequences=1)
print(f"Generated Text:\n{generated_texts[0]}")

prompt = "¿Qué número de tlf tiene Pilar?"
generated_texts = generate_text(prompt, max_length=25, num_return_sequences=1)
print(f"Generated Text:\n{generated_texts[0]}")

prompt = "¿Qué número de tlf tiene Emilia?"
generated_texts = generate_text(prompt, max_length=35, num_return_sequences=1)
print(f"Generated Text:\n{generated_texts[0]}")

Generated Text:
Pilar tiene el número 34. ¿Cuál es el número de su hermano? Pilar tiene un
Generated Text:
¿Qué número de tlf tiene Pilar? Pilar tiene un tlf que empieza con 616 y
Generated Text:
¿Qué número de tlf tiene Emilia? ¿Es un número de teléfono de Madrid? ¿Es un número de teléfono de Barcelona? ¿Es un número


## **Conclusiones**

Vemos que el entrenamiento **autoregressivo** vislumbra un comienzo prometedor al asociar, al menos, el comienzo del número de teléfono correcto. Aún así, el modelo no es capaz de predecir el número de teléfono completo. 

Esto puede deberse a que el modelo no ha sido entrenado en cómo debe responder a números de teléfono. Simplemente genera número que pueden corresponder a un número de teléfono, pero no necesariamente al número de teléfono que se le ha dado en el dataset de entrenamiento.

El modelo tampoco es capaz de terminar la frase correctamente. 